# Parse Email

Parse mbox-format `.txt` files in `sample-data/` into individual emails.

Each file is a thread: messages separated by mbox `From ` envelope lines, each a standard RFC-822 message (`From:`, `To:`, `Date:`, `Subject:` headers + body).


In [2]:
import mailbox
import re
from email.utils import getaddresses
from pathlib import Path

SAMPLE_DIR = Path("..") / "sample-data"

In [3]:
def addrs(header):
    """Extract sorted, lowercased email addresses from a To/From header."""
    if not header:
        return []
    return sorted(a.lower() for _, a in getaddresses([header]) if a)


def normalize(text):
    """Collapse whitespace so near-duplicates compare equal."""
    return re.sub(r"\s+", " ", text).strip()


def parse_file(path):
    """Parse one mbox .txt thread into a list of email dicts."""
    emails = []
    for order, msg in enumerate(mailbox.mbox(str(path))):
        body = msg.get_payload()
        emails.append(
            {
                "doc_id": path.name,
                "canon_order": order,
                "from": addrs(msg["From"]),
                "to": addrs(msg["To"]),
                "subject": msg["Subject"],
                "date": msg["Date"],
                "content": normalize(body),
            }
        )
    return emails

In [4]:
all_emails = []
for path in sorted(SAMPLE_DIR.glob("*.txt")):
    emails = parse_file(path)
    all_emails.extend(emails)
    print(f"{path.name}: {len(emails)} emails")

print(f"\ntotal: {len(all_emails)} emails")

doc1.txt: 1 emails
doc2.txt: 2 emails
doc3.txt: 2 emails
doc4.txt: 3 emails
doc5.txt: 3 emails

total: 11 emails


In [5]:
for e in all_emails:
    print(f"[{e['doc_id']} #{e['canon_order']}] {e['from']} -> {e['to']}")
    print(f"  subject: {e['subject']}")
    print(f"  content: {e['content']}")
    print()

[doc1.txt #0] ['alice@example.com'] -> ['bob@example.com', 'carol@example.com']
  subject: Project kickoff
  content: Hi Bob, Let's schedule the project kickoff for next week. Does Tuesday work for you? Alice

[doc2.txt #0] ['alice@example.com'] -> ['bob@example.com', 'carol@example.com']
  subject: Project kickoff
  content: Hi Bob, Let's schedule the project kickoff for next week. Does Tuesday work for you? Alice

[doc2.txt #1] ['bob@example.com'] -> ['alice@example.com', 'carol@example.com']
  subject: Re: Project kickoff
  content: Hi Alice, Tuesday works great for me. Let's say 10am? Bob

[doc3.txt #0] ['alice@example.com'] -> ['bob@example.com', 'carol@example.com']
  subject: Project kickoff
  content: Hi Bob, Let's schedule the project kickoff for next week. Does Tuesday work for you? Alice

[doc3.txt #1] ['bob@example.com'] -> ['alice@example.com', 'carol@example.com']
  subject: Re: Project kickoff
  content: Hi Alice, Tuesday works great for me. Let's say 10am? Bob

[doc4.tx

In [6]:
all_emails

[{'doc_id': 'doc1.txt',
  'canon_order': 0,
  'from': ['alice@example.com'],
  'to': ['bob@example.com', 'carol@example.com'],
  'subject': 'Project kickoff',
  'date': 'Mon, 1 Jan 2024 09:00:00 +0000',
  'content': "Hi Bob, Let's schedule the project kickoff for next week. Does Tuesday work for you? Alice"},
 {'doc_id': 'doc2.txt',
  'canon_order': 0,
  'from': ['alice@example.com'],
  'to': ['bob@example.com', 'carol@example.com'],
  'subject': 'Project kickoff',
  'date': 'Mon, 1 Jan 2024 09:00:00 +0000',
  'content': "Hi Bob, Let's schedule the project kickoff for next week. Does Tuesday work for you? Alice"},
 {'doc_id': 'doc2.txt',
  'canon_order': 1,
  'from': ['bob@example.com'],
  'to': ['alice@example.com', 'carol@example.com'],
  'subject': 'Re: Project kickoff',
  'date': 'Mon, 1 Jan 2024 10:00:00 +0000',
  'content': "Hi Alice, Tuesday works great for me. Let's say 10am? Bob"},
 {'doc_id': 'doc3.txt',
  'canon_order': 0,
  'from': ['alice@example.com'],
  'to': ['bob@examp

## Rapidfuzz similarity

Try `rapidfuzz` Levenshtein on the parsed email contents. The emails-consumer uses
`Levenshtein.normalized_similarity > 0.9` to flag near-duplicates (after an exact
`content_hash` check). Below we score every pair sharing the same `from`/`to` to see
where the near-dup pairs land.


In [1]:
!pip install rapidfuzz

  Obtaining dependency information for rapidfuzz from https://files.pythonhosted.org/packages/9e/89/c2557e37531d03465193bff0ab9de70b468420a807d71a26a65100635459/rapidfuzz-3.14.5-cp311-cp311-macosx_11_0_arm64.whl.metadata
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 17.9 MB/s eta 0:00:00a 0:00:01

[notice] A new release of pip is available: 23.2.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [ ]:
from rapidfuzz.distance import Levenshtein

# the two known near-duplicate pairs from the sample data
pairs = [
    (
        "doc2#0 vs doc3#0 (whitespace)",
        "Hi Bob, Let's schedule the project kickoff for next week. Does Tuesday work for you? Alice",
        "Hi Bob, Let's schedule the project kickoff for next week. Does Tuesday work for you? Alice",
    ),
    (
        "doc4#2 vs doc5#2 (one word inserted)",
        "Perfect, 10am Tuesday it is. I'll send a calendar invite. Alice",
        "Perfect, 10am on Tuesday it is. I'll send a calendar invite. Alice",
    ),
    (
        "kickoff vs reply (different email)",
        "Hi Bob, Let's schedule the project kickoff for next week. Does Tuesday work for you? Alice",
        "Hi Alice, Tuesday works great for me. Let's say 10am? Bob",
    ),
]

for label, a, b in pairs:
    print(f"{Levenshtein.normalized_similarity(a, b):.3f}  {label}")

1.000  doc2#0 vs doc3#0 (whitespace)
0.955  doc4#2 vs doc5#2 (one word inserted)
0.289  kickoff vs reply (different email)
